# Chronicle LLM v1 — training on a free Kaggle GPU

Trains the ~73M-parameter v1 model with [nanoGPT](https://github.com/karpathy/nanoGPT) on the
tokenised v1 corpus. Repo: https://github.com/ravipatib/ChronicleLLM

**Before running**

1. Settings → Accelerator → **GPU T4 x2** (P100 also works, but set `compile=False` below).
   GPU needs a phone-verified Kaggle account.
2. Settings → **Internet on** (to clone nanoGPT).
3. Add a **private** Kaggle dataset with `train.bin`, `val.bin` and `train_chronicle_v1.py`
   (from `data/australia_1850_1950_v1/` and `config/` after running the build). Keep it private:
   the corpus is built for training, not redistribution.
4. To continue a cut-off run: add the previous run's output (it contains `out-chronicle-v1/ckpt.pt`)
   as a second input. The notebook finds it and resumes.
5. Run with **Save Version → Save & Run All (Commit)** so it keeps running after you close the tab.

Training stops itself after `TIME_LIMIT_H` hours (Kaggle sessions end at ~12h), so the notebook
always finishes cleanly and `/kaggle/working/out-chronicle-v1/ckpt.pt` is saved in the output.

In [ ]:
TIME_LIMIT_H = 11.0      # stop training before Kaggle's ~12h session limit
COMPILE      = True      # False on P100 (torch.compile needs compute capability >= 7.0)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import glob, os, shutil, subprocess, torch
NGPU = torch.cuda.device_count()
print("GPUs:", NGPU)

# nanoGPT pinned to the commit this notebook was tested with
if not os.path.isdir("nanoGPT"):
    !git clone -q https://github.com/karpathy/nanoGPT
    !cd nanoGPT && git checkout -q 3adf61e154c3fe3fca428ad6bc3818b27a3b8291

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    return hits[0] if hits else None

data_dir = "nanoGPT/data/australia_1850_1950_v1"
os.makedirs(data_dir, exist_ok=True)
for f in ["train.bin", "val.bin"]:
    src = find(f)
    assert src, f"{f} not found: add the v1 data as an input dataset"
    if not os.path.exists(f"{data_dir}/{f}"):
        os.symlink(src, f"{data_dir}/{f}")
shutil.copy(find("train_chronicle_v1.py"), "nanoGPT/config/train_chronicle_v1.py")

out_dir = "/kaggle/working/out-chronicle-v1"
os.makedirs(out_dir, exist_ok=True)
prev = [p for p in glob.glob("/kaggle/input/**/out-chronicle-v1/ckpt.pt", recursive=True)]
if prev and not os.path.exists(f"{out_dir}/ckpt.pt"):
    shutil.copy(prev[0], f"{out_dir}/ckpt.pt")
INIT = "resume" if os.path.exists(f"{out_dir}/ckpt.pt") else "scratch"
print("init_from =", INIT)

In [ ]:
import numpy as np
for f in ["train.bin", "val.bin"]:
    n = len(np.memmap(f"{data_dir}/{f}", dtype=np.uint16, mode="r"))
    print(f"{f}: {n/1e6:.1f}M tokens")

## Train

Logs go to `/kaggle/working/train_log.txt` as well as the cell output. `timeout` sends Ctrl-C after
`TIME_LIMIT_H`; nanoGPT has already saved `ckpt.pt` at the last evaluation (every 250 iterations).

In [ ]:
launcher = f"torchrun --standalone --nproc_per_node={NGPU}" if NGPU > 1 else "python"
cmd = (f"cd nanoGPT && timeout --signal=INT {int(TIME_LIMIT_H*3600)} {launcher} train.py "
       f"config/train_chronicle_v1.py --out_dir={out_dir} --init_from={INIT} --compile={COMPILE} "
       f"2>&1 | tee -a /kaggle/working/train_log.txt")
print(cmd)
!{cmd}

## Sample from the checkpoint

In [ ]:
for prompt in ["The drought had lasted three years when the squatter rode into Bourke and",
               "Mr. DEAKIN. — I rise to move that",
               "Q. — What is the price of wool at Geelong? A. —"]:
    !cd nanoGPT && python sample.py --out_dir={out_dir} --start="{prompt}" --num_samples=1 --max_new_tokens=200 --temperature=0.8 --top_k=200 --compile=False

In [ ]:
!grep -E "step [0-9]+: train loss" /kaggle/working/train_log.txt | tail -20
!ls -la {out_dir}